# NLP Project 2

**ESILV A4 DIA6 — 2026**



**Authors:** Leo WINTER & Alvaro SERERO

## Table of Contents   
1. [Setup & Imports](#setup)
2. [Load Data](#load)
3. [Baseline ML model](#model1)
4. [Model with embeddings](#model2)
5. [Model with pre-trained embeddings](#model3)
6. [USE](#model4)
7. [LLM](#model5)
8. [Comparaison](#comparaison)


<a id="setup"></a>
## 1. Setup & Imports

In [2]:
from pathlib import Path


import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns



# Setting up the Paths
CURRENT_DIR = Path.cwd()
DATA_PATH = CURRENT_DIR.parent / "data"
MODEL_PATH = CURRENT_DIR.parent / "model"
VISU_PATH = CURRENT_DIR.parent / "visualizations" / "notebook3"
VISU_PATH.mkdir(parents=True, exist_ok=True)

<a id="load"></a>
## 2. Load Data

In [3]:
# Load data saved in the second notebook
def load_data():
    path_parquet = DATA_PATH / "reviews_modeled.parquet"
    path_csv = DATA_PATH / "reviews_modeled.csv"

    if path_parquet.exists():
        try:
            df = pd.read_parquet(path_parquet)
            return df
        except Exception as e:
            print(e)

    if path_csv.exists():
        try:
            df = pd.read_csv(path_csv)
            return df
        except Exception as e:
            print(e)
    return None


dataset = load_data()
if dataset is None: 
    dataset = pd.DataFrame() #To not have None warnings
display(dataset.head(2))
data_model = dataset.copy()

,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en,year_month,tokens_en,tokens,cluster_en,cluster_fr
0,5,brahim--k-131532,"Meilleurs assurances, prix, solutions, écoute,...",Direct Assurance,auto,train,2021-09-06,2021-09-01,"Best insurance, price, solutions, listening, s...",meilleurs assurances prix solutions écoute rap...,best insurance price solutions listening speed...,2021-09,"['good', 'price', 'solution', 'listen', 'speed...","['meilleur', 'prix', 'solution', 'écoute', 'ra...",1,1
1,4,bernard-g-112497,"je suis globalement satisfait , sauf que vous ...",Direct Assurance,auto,train,2021-05-03,2021-05-01,"I am generally satisfied, except that you have...",je suis globalement satisfait sauf que vous av...,i am generally satisfied except that you have ...,2021-05,"['generally', 'satisfied', 'except', 'problem'...","['globalement', 'satisfait', 'sauf', 'problème...",0,2


In [ ]:
def label_sentiment(rating):
    if rating <= 2: return 0 
    if rating == 3: return 1 
    return 2                 

data_model['emotion'] = data_model['note'].apply(label_sentiment)
X_en = data_model['tokens_en'].apply(lambda x: "".join(x))
X_fr = data_model['tokens'].apply(lambda x: "".join(x))
y_1 = data_model['note']
y_2 = data_model['emotion']

<a id="model1"></a>
## 3. Baseline ML model

In [29]:
def model1(X,y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    tfidf = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=20_000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
    model_lr.fit(X_train_tfidf, y_train)

    y_pred = model_lr.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    print(classification_report(y_test, y_pred))
    
    return tfidf,model_lr,accuracy

In [30]:
print("English model note:")
tfidf_en1, model_lr_en1,accuracy_en1 = model1(X_en,y_1)
print("\nFrench model note:")
tfidf_fr1, model_lr_fr1,accuracy_fr1 = model1(X_fr,y_1)
print("\nEnglish model emotion:")
tfidf_en2, model_lr_en2,accuracy_en2 = model1(X_en,y_2)
print("\nFrench model emotion:")
tfidf_fr2, model_lr_fr2,accuracy_fr2 = model1(X_fr,y_2)

English model note:
              precision    recall  f1-score   support

           1       0.67      0.66      0.66      1453
           2       0.35      0.38      0.36       743
           3       0.28      0.29      0.28       677
           4       0.42      0.38      0.40       977
           5       0.54      0.55      0.54       969

    accuracy                           0.49      4819
   macro avg       0.45      0.45      0.45      4819
weighted avg       0.49      0.49      0.49      4819


French model note:
              precision    recall  f1-score   support

           1       0.67      0.67      0.67      1453
           2       0.36      0.40      0.38       743
           3       0.31      0.29      0.30       677
           4       0.46      0.42      0.44       977
           5       0.56      0.58      0.57       969

    accuracy                           0.50      4819
   macro avg       0.47      0.47      0.47      4819
weighted avg       0.51      0.50    

In [33]:
def plot_top_coefficients(vectorizer, model, n_words=10):
    feature_names = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]
    
    top_pos = np.argsort(coefs)[-n_words:]
    top_neg = np.argsort(coefs)[:n_words]
    
    print("Important negative worlds:", [feature_names[i] for i in top_neg])
    print("Important positive worlds:", [feature_names[i] for i in top_pos])

print("English note:")
print(f"accuracy = {accuracy_en1:.2f}")
plot_top_coefficients(tfidf_en1, model_lr_en1)
print("\nFrench note:")
print(f"accuracy = {accuracy_fr1:.2f}")
plot_top_coefficients(tfidf_fr1, model_lr_fr1)
print("\nEnglish emotion:")
print(f"accuracy = {accuracy_en2:.2f}")
plot_top_coefficients(tfidf_en2, model_lr_en2)
print("\nFrench emotion:")
print(f"accuracy = {accuracy_fr2:.2f}")
plot_top_coefficients(tfidf_fr2, model_lr_fr2)

English note:
accuracy = 0.49
Important negative worlds: ['satisfied', 'thank', 'good', 'price', 'fast', 'moment', 'quick', 'easy', 'hope', 'satisfied service']
Important positive worlds: ['expensive', 'impossible', 'reimburse', 'pay', 'terminate', 'month', 'incompetent', 'increase', 'avoid', 'flee']

French note:
accuracy = 0.50
Important negative worlds: ['satisfait', 'satisfaite', 'rapide', 'merci', 'prix', 'très', 'espère', 'écoute', 'bon', 'efficace']
Important positive worlds: ['argent', 'incompétent', 'nul', 'aucun', 'déconseille', 'aucune', 'éviter', 'fuyez', 'mois', 'fuir']

English emotion:
accuracy = 0.75
Important negative worlds: ['satisfied', 'thank', 'good', 'fast', 'quick', 'effective', 'easy', 'responsive', 'well', 'practical']
Important positive worlds: ['pay', 'refuse', 'increase', 'incompetent', 'impossible', 'deplorable', 'avoid', 'month', 'terminate', 'flee']

French emotion:
accuracy = 0.78
Important negative worlds: ['satisfait', 'rapide', 'satisfaite', 'merci',

<a id="model2"></a>
## 4. Model with embeddings

<a id="model3"></a>
## 5. Model with pre-trained embeddings

<a id="model4"></a>
## 6. USE

<a id="model5"></a>
## 7. LLM

<a id="comparaison"></a>
## 8. Comparaison